# Understanding Font File Structure / フォントファイルの構造を理解する

A TTF/OTF font file is a **collection of tables**, each responsible for specific information.
This notebook walks through the key tables using `fontTools`.

In [1]:
from fontTools.ttLib import TTFont

FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = TTFont(FONT_PATH, lazy=True)
print("Loaded:", FONT_PATH)

Loaded: /usr/share/fonts/truetype/dejavu/DejaVuSans.ttf


## 1. Table List / テーブル一覧

`TTFont` behaves like a dictionary: keys are 4-character table tags, values are parsed table objects.
/ `TTFont` は辞書ライクなオブジェクト。キーがテーブルタグ（4文字）、値が解析済みオブジェクト。

In [2]:
tables = font.keys()
print(f"Table count: {len(tables)}")
print(sorted(tables))

Table count: 21
['FFTM', 'GDEF', 'GPOS', 'GSUB', 'GlyphOrder', 'MATH', 'OS/2', 'cmap', 'cvt ', 'fpgm', 'gasp', 'glyf', 'head', 'hhea', 'hmtx', 'kern', 'loca', 'maxp', 'name', 'post', 'prep']


Key tables and their roles / 主なテーブルの役割:

| Tag | Content |
|-----|---------|
| `name` | Font name, version, license strings |
| `OS/2` | Weight, width, Unicode range |
| `cmap` | Unicode codepoint → glyph name mapping |
| `post` | Monospaced flag, italic angle, PostScript name |
| `head` | Font version, creation date, units per em |
| `glyf` | Glyph outline data (Bézier curves) |
| `hhea` | Horizontal metrics (ascent / descent) |

## 2. `name` Table — Font Metadata / フォントのメタデータ

The `name` table is a list of `(nameID, string)` pairs.
The same nameID can appear for multiple languages and platforms.
/ 同じ nameID が複数の言語・プラットフォーム向けに格納されていることがある。

In [3]:
NAME_IDS = {
    0: "copyright",
    1: "family",
    2: "subfamily",
    3: "unique_id",
    4: "full_name",
    5: "version",
    6: "postscript_name",
    7: "trademark",
    8: "manufacturer",
    9: "designer",
    13: "license",
    14: "license_url",
}

for nid, label in NAME_IDS.items():
    # prefer Windows/BMP/English; fall back to Mac
    record = font["name"].getName(nid, 3, 1, 0x0409)
    if record is None:
        record = font["name"].getName(nid, 1, 0, 0)
    value = record.toUnicode() if record else None
    # truncate long license strings
    if value and len(value) > 50:
        value = value[:50] + "..."
    print(f"  [{nid:2d}] {label:20s}: {value}")

  [ 0] copyright           : Copyright (c) 2003 by Bitstream, Inc. All Rights R...
  [ 1] family              : DejaVu Sans
  [ 2] subfamily           : Book
  [ 3] unique_id           : DejaVu Sans
  [ 4] full_name           : DejaVu Sans
  [ 5] version             : Version 2.37
  [ 6] postscript_name     : DejaVuSans
  [ 7] trademark           : None
  [ 8] manufacturer        : DejaVu fonts team
  [ 9] designer            : None
  [13] license             : Fonts are (c) Bitstream (see below). DejaVu change...
  [14] license_url         : http://dejavu.sourceforge.net/wiki/index.php/Licen...


## 3. `OS/2` Table — Weight, Width, Style / 太さ・幅・スタイル

Defines the visual attributes of the font. / フォントの視覚的な属性を定義する。

In [4]:
os2 = font["OS/2"]

WEIGHT_NAMES = {
    100: "Thin", 200: "ExtraLight", 300: "Light", 400: "Regular",
    500: "Medium", 600: "SemiBold", 700: "Bold", 800: "ExtraBold", 900: "Black"
}
WIDTH_NAMES = {
    1: "UltraCondensed", 2: "ExtraCondensed", 3: "Condensed", 4: "SemiCondensed",
    5: "Normal", 6: "SemiExpanded", 7: "Expanded", 8: "ExtraExpanded", 9: "UltraExpanded"
}

print(f"weight_class : {os2.usWeightClass} ({WEIGHT_NAMES.get(os2.usWeightClass, '?')})")
print(f"width_class  : {os2.usWidthClass}  ({WIDTH_NAMES.get(os2.usWidthClass, '?')})")

weight_class : 400 (Regular)
width_class  : 5  (Normal)


## 4. `post` Table — Monospaced Flag, Italic Angle / 等幅・イタリック角度

In [5]:
post = font["post"]

print(f"is_monospaced : {bool(post.isFixedPitch)}")
print(f"italic_angle  : {post.italicAngle}")

is_monospaced : False
italic_angle  : 0.0


## 5. `cmap` Table — Codepoint → Glyph Mapping / コードポイント→グリフのマッピング

`cmap` defines which characters the font supports.
Keys are Unicode codepoints (int), values are glyph names (str).
/ キーが Unicode コードポイント（整数）、値がグリフ名（文字列）。

In [6]:
cmap = font.getBestCmap()  # pick the best available cmap subtable

print(f"Supported codepoints: {len(cmap)}")
print()

# first 10 entries
print("Codepoint → glyph name (first 10):")
for cp, glyph_name in list(cmap.items())[:10]:
    char = chr(cp)
    print(f"  U+{cp:04X}  '{char}'  → {glyph_name}")

Supported codepoints: 5918

Codepoint → glyph name (first 10):
  U+0020  ' '  → space
  U+0021  '!'  → exclam
  U+0022  '"'  → quotedbl
  U+0023  '#'  → numbersign
  U+0024  '$'  → dollar
  U+0025  '%'  → percent
  U+0026  '&'  → ampersand
  U+0027  '''  → quotesingle
  U+0028  '('  → parenleft
  U+0029  ')'  → parenright


In [7]:
# check glyph existence for specific characters
test_chars = ["A", "a", "あ", "漢", "α", "→"]

print("Glyph existence check:")
for c in test_chars:
    has = ord(c) in cmap
    mark = "✓" if has else "✗"
    print(f"  {mark} U+{ord(c):04X} '{c}'")

Glyph existence check:
  ✓ U+0041 'A'
  ✓ U+0061 'a'
  ✗ U+3042 'あ'
  ✗ U+6F22 '漢'
  ✓ U+03B1 'α'
  ✓ U+2192 '→'


## 6. `head` Table — Basic Info / 基本情報

In [8]:
head = font["head"]

print(f"units_per_em  : {head.unitsPerEm}")
print(f"font_revision : {head.fontRevision}")

units_per_em  : 2048
font_revision : 2.3699951171875


`units_per_em` defines the precision of the font's coordinate system (typically 1000 or 2048).
Glyph outlines are described in this unit and scaled to actual pixels at render time.
/ グリフの輪郭データはこの単位系で記述され、レンダリング時に実ピクセルへスケールされる。

## 7. Comparing Multiple Fonts / 複数フォントの比較

Verify that `weight_class` changes between Regular and Bold.
/ Regular と Bold で `weight_class` がどう変わるか確認する。

In [9]:
font_paths = {
    "DejaVuSans": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "DejaVuSans-Bold": "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
}

print(f"{'name':20s}  {'weight':>6}  {'monospaced':>10}  {'glyphs':>6}")
print("-" * 50)

for name, path in font_paths.items():
    f = TTFont(path, lazy=True)
    w = f["OS/2"].usWeightClass
    mono = bool(f["post"].isFixedPitch)
    n = len(f.getBestCmap() or {})
    print(f"{name:20s}  {w:>6}  {str(mono):>10}  {n:>6}")

name                  weight  monospaced  glyphs
--------------------------------------------------
DejaVuSans               400       False    5918
DejaVuSans-Bold          700       False    5898


## Summary / まとめ

| Table | Information | Use case |
|-------|-------------|----------|
| `name`  | Family name, version, license | Metadata collection |
| `OS/2`  | Weight, width | Font classification |
| `post`  | Monospaced flag, italic angle | Font classification |
| `cmap`  | Supported codepoints | Glyph existence check |
| `head`  | Units per em | Rendering calculation |

These fields are aggregated into `FontMetadata` by `fontreader.py`.